# Vault AWS Secrets Engine - AccessKey 방식

이 노트북은 **IAM User의 AccessKey/SecretKey**를 Vault에 등록하여 AWS에 인증하는 방식을 테스트합니다.

HashiCorp 공식 문서 - https://developer.hashicorp.com/vault/docs/secrets/aws

---

## 사전 준비

- Vault가 AWS에 인증할 **IAM User**가 필요합니다.
- IAM User에 동적 자격증명 발급을 위한 권한 부여:
  - `iam:CreateUser`, `iam:AttachUserPolicy`, `iam:CreateAccessKey`, `iam:DeleteUser` 등
  - 또는 간단하게 `IAMFullAccess` 정책 부여
- `.envrc`에 `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION` 설정

## 흐름

```
1. Enable  → AWS Secrets Engine 마운트
2. Config  → IAM User 자격증명(AccessKey) 등록
3. Role    → Vault Role 생성 (어떤 AWS 권한으로 발급할지)
4. Creds   → 동적 자격증명 발급
5. Lease   → Lease 관리 (갱신/취소)
6. Cleanup → Role 삭제 및 Engine 언마운트
```

## 환경변수 설정

In [ ]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

# Vault AWS Secrets Engine 마운트 경로
export MOUNT_PATH="my-aws-accesskey"

echo "VAULT_ADDR            : $VAULT_ADDR"
echo "VAULT_TOKEN           : ${VAULT_TOKEN:0:10}..."
echo "MOUNT_PATH            : $MOUNT_PATH"
echo "AWS_REGION            : $AWS_REGION"
echo "AWS_ACCESS_KEY_ID     : ${AWS_ACCESS_KEY_ID:0:6}..."
echo "AWS_SECRET_ACCESS_KEY : ${AWS_SECRET_ACCESS_KEY:0:6}..."

## 1. AWS Secrets Engine 마운트

In [ ]:
# AWS Secrets Engine 마운트
vault secrets enable \
    -path="$MOUNT_PATH" \
    -default-lease-ttl=5m \
    -max-lease-ttl=2h \
    aws

## 2. Root Config 설정 (AccessKey 방식)

IAM User의 AccessKey/SecretKey를 Vault에 등록합니다.  
Vault는 이 자격증명으로 AWS API를 호출하여 동적 자격증명을 발급합니다.

In [ ]:
# AccessKey 방식으로 Vault에 AWS 자격증명 등록
vault write "$MOUNT_PATH/config/root" \
  access_key="$AWS_ACCESS_KEY_ID" \
  secret_key="$AWS_SECRET_ACCESS_KEY" \
  region="$AWS_REGION"

In [ ]:
# 등록된 root config 확인 (secret_key는 마스킹됨)
vault read "$MOUNT_PATH/config/root"

## 3. Vault Role 생성

Vault Role은 **어떤 AWS 권한으로** 동적 자격증명을 발급할지 정의합니다.

| `credential_type` | 설명 |
|---|---|
| `iam_user` | IAM User + AccessKey를 동적으로 생성 |
| `assumed_role` | STS AssumeRole로 임시 자격증명 발급 |
| `federation_token` | STS GetFederationToken으로 임시 자격증명 발급 |

In [ ]:
# Vault Role 생성 - credential_type: iam_user
# IAM User와 AccessKey를 동적으로 생성하여 발급합니다.
vault write "$MOUNT_PATH/roles/my-iam-user-role" \
  credential_type=iam_user \
  policy_arns="arn:aws:iam::aws:policy/SecretsManagerReadWrite"

In [ ]:
# Vault Role 생성 - credential_type: assumed_role
# STS AssumeRole을 통해 임시 자격증명(STS Token)을 발급합니다.
vault write "$MOUNT_PATH/roles/my-sts-role" \
  credential_type=assumed_role \
  role_arns="$VAULT_ASSUME_TARGET_ROLE_ARN" \
  default_sts_ttl=15m \
  max_sts_ttl=12h

In [ ]:
# 생성된 Role 목록 확인
echo "=== Role 목록 ==="
vault list "$MOUNT_PATH/roles"

echo ""
echo "=== my-iam-user-role 상세 ==="
vault read "$MOUNT_PATH/roles/my-iam-user-role"

echo ""
echo "=== my-sts-role 상세 ==="
vault read "$MOUNT_PATH/roles/my-sts-role"

## 4. 동적 자격증명 발급 (Generate Credentials)

Vault가 AWS에 IAM User 또는 STS Token을 **동적으로 생성**합니다.

In [ ]:
# IAM User 자격증명 발급 (iam_user)
echo "=== IAM User 자격증명 발급 ==="
vault read "$MOUNT_PATH/creds/my-iam-user-role"

In [ ]:
# STS 임시 자격증명 발급 (assumed_role)
echo "=== STS 임시 자격증명 발급 ==="
vault read "$MOUNT_PATH/sts/my-sts-role" ttl=15m

In [ ]:
# JSON 형식으로 발급 후 특정 필드 추출 (IAM User 및 STS 모두 추출)
echo "=== 1. IAM User 자격증명 발급 및 추출 ==="
IAM_CREDS=$(vault read -format=json "$MOUNT_PATH/creds/my-iam-user-role")
export IAM_LEASE_ID=$(echo $IAM_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")
echo "IAM Lease ID    : $IAM_LEASE_ID"

echo ""
echo "=== 2. STS 임시 자격증명 발급 및 추출 ==="
STS_CREDS=$(vault read -format=json "$MOUNT_PATH/sts/my-sts-role")
export STS_LEASE_ID=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")
DYNA_ACCESS_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('access_key', ''))")
DYNA_SECRET_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('secret_key', ''))")
DYNA_SESSION_TOKEN=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('security_token', d['data'].get('session_token', '')))")
echo "STS Lease ID    : $STS_LEASE_ID"
echo "STS AccessKeyId : $DYNA_ACCESS_KEY"

In [ ]:
# 발급된 동적 자격증명을 환경 변수에 할당
export AWS_ACCESS_KEY_ID="$DYNA_ACCESS_KEY"
export AWS_SECRET_ACCESS_KEY="$DYNA_SECRET_KEY"
export AWS_SESSION_TOKEN="$DYNA_SESSION_TOKEN"
export AWS_DEFAULT_REGION="$AWS_REGION"

# 1. 현재 인증된 자격증명 정보 확인
echo "=== 1. STS GetCallerIdentity 호출 ==="
aws sts get-caller-identity

In [ ]:
# 2. Secrets Manager에 테스트용 시크릿 생성
echo "=== 2. SecretsManager CreateSecret 호출 ==="
aws secretsmanager create-secret --name test-key

In [ ]:
# 3. S3 버킷 목록 조회 (권한이 없으므로 오류)
echo "=== 3. S3 ListBuckets 호출 ==="
aws s3 ls

In [ ]:
# 4. 생성한 테스트 시크릿 즉시 삭제 (복구 기간 없이 강제 삭제)
echo "=== 4. SecretsManager DeleteSecret 호출 ==="
aws secretsmanager delete-secret --force-delete-without-recovery --secret-id test-key

## 5. Lease 관리 (갱신 / 취소)

발급된 동적 자격증명은 **Lease**로 관리됩니다.  

In [ ]:
# Lease 목록 확인
echo "=== IAM User Role Lease 목록 ==="
vault list sys/leases/lookup/${MOUNT_PATH}/creds/my-iam-user-role/ || echo "(활성 Lease 없음)"

echo ""
echo "=== STS Role Lease 목록 ==="
vault list sys/leases/lookup/${MOUNT_PATH}/sts/my-sts-role/ || echo "(활성 Lease 없음)"

In [ ]:
# lease ID 조회
vault lease lookup ${IAM_LEASE_ID}
echo ""
vault lease lookup ${STS_LEASE_ID}

In [ ]:
# Lease 갱신 (갱신 가능한 IAM User 방식 테스트)
echo "=== IAM User Lease 갱신 (60분 연장 시도) ==="
vault lease renew -increment=3600 "$IAM_LEASE_ID"

In [ ]:
# 특정 Lease 취소 (해당 IAM User/STS Token 즉시 삭제)
echo "=== Lease 취소 (자격증명 즉시 폐기) ==="
# 예: IAM User 리스 취소
vault lease revoke "$IAM_LEASE_ID"

In [ ]:
# Role에 속한 모든 Lease 일괄 취소
echo "=== my-iam-user-role 의 모든 Lease 취소 ==="
vault lease revoke -prefix "${MOUNT_PATH}/creds/my-iam-user-role"

## 6. (정리) AWS Secrets Engine 언마운트

In [ ]:
vault secrets disable "$MOUNT_PATH"

echo ""
vault secrets list